In [0]:
ECOMM_SILVER_FILES = "abfss://silver@gksdatalake4.dfs.core.windows.net/ecomm/"

ECOMM_GOLD_COUNTRY_AGGREGATE_FILES = "abfss://gold@gksdatalake4.dfs.core.windows.net/ecomm/"

ECOMM_GOLD_PRODUCT_AGGREGATE_FILES = "abfss://gold@gksdatalake4.dfs.core.windows.net/ecomm/"

CHECKPOINT_COUNTRY_LOCATION = "abfss://bronze@gksdatalake4.dfs.core.windows.net/checkpoints/ecom-stream-gold-country-aggregate"

CHECKPOINT_PRODUCT_LOCATION = "abfss://bronze@gksdatalake4.dfs.core.windows.net/checkpoints/ecom-stream-gold-product-aggregate"

In [0]:
df = (
    spark.readStream
    .format("delta")
    .load(ECOMM_SILVER_FILES)
)

df.printSchema() 

In [0]:
import pyspark.sql.functions as F
# to ignore late coming data.. after df. but our data is older than 14 years..it will reject all data.
# .withWatermark("InvoiceDate", "2 hours")  # Optional: ignore if you want to keep late data always

# Hourly country wise sales
country_hourly_sales_df = (
    df
    # for water mark placeholder for live data
    .groupBy(
        F.window("InvoiceDate", "1 hour"),  # group per hour
        F.col("Country")
    )
    .agg(
        F.sum("Amount").alias("TotalSales")
    )
    .select(
       
        F.col("window.start").alias("HourStart"),
        F.col("window.end").alias("HourEnd"),
        "Country",
        "TotalSales"
    )
)

country_hourly_sales_df.printSchema()

In [0]:
# hourly product sales
# .withWatermark("InvoiceDate", "2 hours")  # Optional

product_hourly_sales_df = (
    df
    
    .groupBy(
        F.window("InvoiceDate", "1 hour"),
        F.col("StockCode")
    )
    .agg(
        F.sum("Quantity").alias("TotalQuantity"),
        F.sum("Amount").alias("TotalSales")
    )
    .select(
        F.col("window.start").alias("HourStart"),
        F.col("window.end").alias("HourEnd"),
        "StockCode",
        "TotalQuantity",
        "TotalSales"
    )
)


In [0]:
%sql

-- assuming we have deloitte_db catalog, gksdb database/namespace
CREATE OR REPLACE TABLE aon_impact.gksdb.country_hourly_sales
(
  HourStart TIMESTAMP,
  HourEnd TIMESTAMP,
  Country STRING,
  TotalSales DOUBLE 
)
USING DELTA
 LOCATION 'abfss://gold@gksdatalake4.dfs.core.windows.net/aon-ecomm-aggregate-country-hourly-sales';

In [0]:
def upsert_to_delta_country_hourly_sales(microBatchDF, batch_id):
    from delta.tables import DeltaTable

    target_table = DeltaTable.forName(spark, "aon_impact.gksdb.country_hourly_sales")

    (target_table.alias("target")
     .merge(
         microBatchDF.alias("source"),
         "target.Country = source.Country AND target.HourStart = source.HourStart"
     )
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())

In [0]:
country_sales_query = country_hourly_sales_df.writeStream \
    .outputMode("update") \
    .option("checkpointLocation", CHECKPOINT_COUNTRY_LOCATION) \
    .foreachBatch(upsert_to_delta_country_hourly_sales) \
    .start()